In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=YMSC2dBCwERLXKUj1cNcmUA7mlOiIT&access_type=offline&code_challenge=9pcSgUX5aikxx5jTyf9TG0OHXsZo8npDaJhZ9h2bI-I&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Clo

In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/06 21:16:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
path_to_release_folder="gs://open-targets-data-releases/25.06/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

sl_eff=session.spark.read.parquet("gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect")

l2g_full=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_prioritised_genes_per_CS.parquet")

In [3]:
variant_pleiotropy = session.spark.read.parquet('gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/variant_pleiotropy')

In [6]:
variant_pleiotropy.count()

40706

In [7]:
variant_pleiotropy.printSchema()

root
 |-- variantId: string (nullable = true)
 |-- nonEURProportion: double (nullable = true)
 |-- minAbsBeta: double (nullable = true)
 |-- maxAbsBeta: double (nullable = true)
 |-- minCoefficientDetermination: double (nullable = true)
 |-- maxCoefficientDetermination: double (nullable = true)
 |-- minEffectiveSampleSize: double (nullable = true)
 |-- maxEffectiveSampleSize: double (nullable = true)
 |-- minVarG: double (nullable = true)
 |-- maxVarG: double (nullable = true)
 |-- maxMAF: double (nullable = true)
 |-- averageBetaSign: double (nullable = true)
 |-- stdBetaSign: double (nullable = true)
 |-- betaSignConcordance: double (nullable = true)
 |-- earliestPublicationDate: string (nullable = true)
 |-- uniqueDiseases: integer (nullable = true)
 |-- uniqueTherapeuticAreas: integer (nullable = true)
 |-- cancerOrBenignTumor: long (nullable = true)
 |-- infectiousDisease: long (nullable = true)
 |-- pregnancyOrPerinatalDisease: long (nullable = true)
 |-- disorderOfVisualSystem: 

In [4]:
variant_pleiotropy.filter(f.col("uniqueDiseases")>=10).count()

197

In [9]:
variant_pleiotropy.filter(f.col("uniqueTherapeuticAreas")>1).count()

5900

In [12]:
# collect to pandas if it fits in driver memory
pdf = (
    variant_pleiotropy
    .select("uniqueTherapeuticAreas", "uniqueDiseases")
    .na.drop()
    .toPandas()
)
pdf["uniqueTherapeuticAreas"] = pdf["uniqueTherapeuticAreas"].astype(float)
pdf["uniqueDiseases"] = pdf["uniqueDiseases"].astype(float)

from scipy.stats import pearsonr, spearmanr
pearson_r, pearson_p = pearsonr(pdf["uniqueTherapeuticAreas"], pdf["uniqueDiseases"])
spearman_r, spearman_p = spearmanr(pdf["uniqueTherapeuticAreas"], pdf["uniqueDiseases"])

print("Pearson r, p-value:", pearson_r, pearson_p)
print("Spearman rho, p-value:", spearman_r, spearman_p)

Pearson r, p-value: 0.7635021976212528 0.0
Spearman rho, p-value: 0.7516418172256819 0.0


25/11/06 15:35:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 155341 ms exceeds timeout 120000 ms
25/11/06 15:35:44 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/06 15:35:46 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

In [3]:
x=session.spark.read.parquet('gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/pleiotropic_discordant_variants')

In [5]:
x.count()

52

In [6]:
x.printSchema()

root
 |-- variantId: string (nullable = true)
 |-- betaSignConcordance: double (nullable = true)
 |-- uniqueDiseases: integer (nullable = true)
 |-- uniqueTherapeuticAreas: integer (nullable = true)
 |-- uniqueDiseaseNames: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- uniqueTherapeuticAreaNames: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- prioritisedGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [7]:
x.show(1)

+---------------+-------------------+--------------+----------------------+--------------------+--------------------------+-----------------+
|      variantId|betaSignConcordance|uniqueDiseases|uniqueTherapeuticAreas|  uniqueDiseaseNames|uniqueTherapeuticAreaNames| prioritisedGenes|
+---------------+-------------------+--------------+----------------------+--------------------+--------------------------+-----------------+
|19_44908684_T_C| 0.6594594594594595|            85|                    15|[MONDO_0021187, E...|      [OTAR_0000020, EF...|[ENSG00000130203]|
+---------------+-------------------+--------------+----------------------+--------------------+--------------------------+-----------------+
only showing top 1 row



In [8]:
x.select("prioritisedGenes").distinct().count()

41

In [14]:
x_exploded=x.withColumn("gene",f.explode("prioritisedGenes"))

In [15]:
x_exploded.count()

60

In [16]:
x_exploded.select("gene").distinct().count()

44

In [ ]:
x.toPandas().to_csv("./data/antagonistic_pl.csv",index=False)

25/11/06 19:25:39 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 220041 ms exceeds timeout 120000 ms
25/11/06 19:25:39 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/06 19:25:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [9]:
target=session.spark.read.parquet("gs://open-targets-pipeline-runs/il/25.09-testrun-1/output/target")

In [10]:
target.printSchema()

root
 |-- id: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- biotype: string (nullable = true)
 |-- transcriptIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- canonicalTranscript: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: string (nullable = true)
 |-- canonicalExons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genomicLocation: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: integer (nullable = true)
 |-- alternativeGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- approvedName: string (nullable = true)
 |-- go: array (nullable = true)
 |    |-- element: struct (containsNull = tru

In [11]:
target=target.select("approvedSymbol","id").withColumnRenamed("id","prioritisedGenes")

In [13]:
target.show(1)

+--------------+----------------+
|approvedSymbol|prioritisedGenes|
+--------------+----------------+
|        TSPAN6| ENSG00000000003|
+--------------+----------------+
only showing top 1 row



In [12]:
x=x.join(target,"prioritisedGenes","inner").cache()
x.count()

25/11/06 17:59:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve "(prioritisedGenes = prioritisedGenes)" due to data type mismatch: the left and right operands of the binary operator have incompatible types ("ARRAY<STRING>" and "STRING").;
'Project [prioritisedGenes#172, variantId#166, betaSignConcordance#167, uniqueDiseases#168, uniqueTherapeuticAreas#169, uniqueDiseaseNames#170, uniqueTherapeuticAreaNames#171, approvedSymbol#230]
+- 'Join Inner, (prioritisedGenes#172 = prioritisedGenes#290)
   :- Relation [variantId#166,betaSignConcordance#167,uniqueDiseases#168,uniqueTherapeuticAreas#169,uniqueDiseaseNames#170,uniqueTherapeuticAreaNames#171,prioritisedGenes#172] parquet
   +- Project [approvedSymbol#230, id#229 AS prioritisedGenes#290]
      +- Project [approvedSymbol#230, id#229]
         +- Relation [id#229,approvedSymbol#230,biotype#231,transcriptIds#232,canonicalTranscript#233,canonicalExons#234,genomicLocation#235,alternativeGenes#236,approvedName#237,go#238,hallmarks

In [6]:
vlp=session.spark.read.parquet("./data/varaint_effects_diseases")

In [7]:
vlp.count()

77405

In [8]:
vlp.select("variantId").distinct().count()

40706

In [9]:
vlp.select("diseaseId").distinct().count()

1403

In [10]:
vlp.show(1)

+--------------------+--------------------+---------------+--------------------+------+-------------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------------+------------------+------------------+-------------+
|             studyId|        studyLocusId|      variantId|             variant|geneId|       originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|traitFromSourceMappedIds|  absEstimatedBeta|               maf|    diseaseId|
+--------------------+--------------------+---------------+--------------------+------+-------------------+---------------------+--------------------+-----------------+----------+-------------------

In [11]:
from pyspark.sql.window import Window

# Create window partitioned by variantId and studyId, ordered by absEstimatedBeta descending
window = Window.partitionBy("variantId", "studyId").orderBy(f.desc("absEstimatedBeta"))

# Keep only the row with the highest absEstimatedBeta for each variantId-studyId combination
vlp_deduplicated = vlp.withColumn("row_num", f.row_number().over(window)) \
                      .filter(f.col("row_num") == 1) \
                      .drop("row_num")

In [12]:
from pyspark.sql.window import Window

# Create a window partitioned by diseaseId and variantId, ordered by absEstimatedBeta descending
window = Window.partitionBy("diseaseId", "variantId").orderBy(f.desc("absEstimatedBeta"))

# Add row number and filter for the first row (max absEstimatedBeta)
vlp_beta_max = vlp_deduplicated.withColumn("row_num", f.row_number().over(window)) \
             .filter(f.col("row_num") == 1) \
             .drop("row_num") \
             .cache()

vlp_beta_max.count()

54728

In [13]:
# Count unique diseases per variant
ps = vlp_deduplicated.groupBy("variantId").agg(
    f.countDistinct("diseaseId").alias("uniqueDiseases")
)

# Show the result
ps.show()

+----------------+--------------+
|       variantId|uniqueDiseases|
+----------------+--------------+
| 1_176005154_T_C|             1|
| 17_71220546_T_C|             1|
|  9_35522667_A_G|             1|
| 3_107499086_T_A|             1|
|10_121156039_T_C|             2|
|  6_20377383_C_T|             1|
|12_109806478_T_C|             2|
| 18_33833715_C_T|             1|
| X_130238143_G_C|             1|
|  9_21998661_G_A|             1|
|10_62672213_CT_C|             1|
| 6_126434496_T_G|             1|
|  19_7200979_C_T|             1|
|  6_19377102_A_G|             1|
| 6_133947684_G_A|             1|
|  9_21973423_G_T|             2|
| 3_128179658_A_C|             1|
|12_111440722_T_C|             1|
| 16_53796143_C_G|             1|
|  4_81665896_A_G|             2|
+----------------+--------------+
only showing top 20 rows



In [14]:
ps.filter(f.col("uniqueDiseases")>10).show()

+----------------+--------------+
|       variantId|uniqueDiseases|
+----------------+--------------+
|  6_36679512_G_A|            11|
|11_116778201_G_C|            21|
| 16_53772541_A_G|            15|
| 7_130023656_C_T|            11|
|  2_43845437_G_T|            13|
|   5_1287079_G_A|            16|
|   5_1279675_C_T|            22|
|  9_22103814_A_G|            12|
| 16_53769311_G_A|            13|
| 12_57133500_T_C|            16|
|  9_22124505_A_T|            16|
| 22_43928850_C_T|            18|
| 2_162267541_C_T|            15|
|12_111395984_G_A|            12|
| 19_33235671_G_A|            11|
| 11_76582682_G_T|            11|
| 4_102267552_C_T|            41|
| 16_20381010_G_A|            14|
| 8_125487789_C_G|            12|
| 7_150993088_C_T|            20|
+----------------+--------------+
only showing top 20 rows



In [15]:
vlp_beta_max_high_ps=vlp_beta_max.join(ps.filter(f.col("uniqueDiseases")>10).select("variantId"),"variantId","inner")
vlp_beta_max_high_ps.count()

2153

In [16]:
from scipy.stats import gamma
import pandas as pd
import numpy as np

# Assume spark session exists
# spark = SparkSession.builder.getOrCreate()

# Define a UDF to fit gamma and return shape, scale
def fit_gamma_udf(values):
    values = np.array(values)
    values = values[values > 0]  # Gamma requires positive values
    if len(values) < 10:
        return (None, None)
    # Fit gamma: use method-of-moments or MLE
    # scipy.stats.gamma.fit returns (shape, loc, scale)
    try:
        shape, loc, scale = gamma.fit(values, floc=0)
        return float(shape), float(scale)
    except Exception:
        return (None, None)

from pyspark.sql.types import StructType, StructField, DoubleType

gamma_schema = StructType([
    StructField("shape", DoubleType(), True),
    StructField("scale", DoubleType(), True)
])

# Collect absEstimatedBeta list per variantId
beta_grouped = (vlp_beta_max_high_ps
                .groupBy("variantId")
                .agg(f.collect_list(f.col("absEstimatedBeta")).alias("beta_list")))

# Register UDF
fit_gamma_spark_udf = f.udf(fit_gamma_udf, gamma_schema)

# Fit gamma on absEstimatedBeta
gamma_fitted = beta_grouped.withColumn("gamma_fit_beta", fit_gamma_spark_udf("beta_list"))

# Extract shape/scale
gamma_fitted = gamma_fitted.select(
    "variantId",
    gamma_fitted["gamma_fit_beta"].shape.alias("beta_shape"),
    gamma_fitted["gamma_fit_beta"].scale.alias("beta_scale")
)

# Now also fit gamma for absEstimatedBeta^2
def fit_gamma_squared_udf(values):
    values = np.array(values)
    values = values[values > 0]  # gamma positive
    values = values ** 2
    if len(values) < 10:
        return (None, None)
    try:
        shape, loc, scale = gamma.fit(values, floc=0)
        return float(shape), float(scale)
    except Exception:
        return (None, None)

fit_gamma_sq_udf = f.udf(fit_gamma_squared_udf, gamma_schema)

gamma_fitted = gamma_fitted.join(
    beta_grouped.withColumn("gamma_fit_beta2", fit_gamma_sq_udf("beta_list"))
                .select("variantId",
                        f.col("gamma_fit_beta2").shape.alias("beta2_shape"),
                        f.col("gamma_fit_beta2").scale.alias("beta2_scale")),
    on="variantId"
)

# Collect to pandas
gamma_params_df = gamma_fitted.toPandas()


In [17]:
gamma_params_df

,variantId,beta_shape,beta_scale,beta2_shape,beta2_scale
0,5_1286401_C_A,1.368556,0.220707,0.496392,0.292685
1,2_102319699_T_A,9.228355,0.016719,2.254006,0.011999
2,12_111569952_C_T,3.794427,0.017385,0.972369,0.006166
3,1_113761186_C_A,2.016312,0.163392,0.666888,0.237130
4,5_111066174_T_C,2.615563,0.059133,0.725366,0.050328
...,...,...,...,...,...
113,20_33950585_G_C,3.105251,0.088427,1.015774,0.093242
114,15_27985172_C_T,7.109712,0.023085,1.974074,0.015544
115,16_53769662_T_A,5.147720,0.014265,1.468838,0.004382
116,1_152312600_CACTG_C,3.706280,0.108171,1.078367,0.191155


In [18]:
vlp_beta_max_high_ps.filter(f.col("variantId")=="9_22124745_C_G").select("absEstimatedBeta").show(100)

+--------------------+
|    absEstimatedBeta|
+--------------------+
| 0.07883535278110519|
|0.030872661279000867|
| 0.05363171690758647|
| 0.03850553581657284|
| 0.13520505194742938|
| 0.06502301093338424|
| 0.07489795645306359|
| 0.04365983747099578|
| 0.11083026574244388|
| 0.12445843669420355|
| 0.07684962939896704|
| 0.11580216127797223|
| 0.22287798151967794|
| 0.05991694265080245|
|  0.1888225775300826|
| 0.03891898167610717|
| 0.05491866198626455|
| 0.20340966480076852|
|  0.2052254670613944|
+--------------------+



In [19]:
import numpy as np
from sklearn.mixture import GaussianMixture

# Initialize output array
out = np.full((1,6), np.nan)

# Original |β| values
vals = np.array([
    0.07883535278110519,0.030872661279000867,0.03891898167610717,
    0.05363171690758647,0.03850553581657284,0.13520505194742938,
    0.06502301093338424,0.07489795645306359,0.04365983747099578,
    0.11083026574244388,0.12445843669420355,0.07684962939896704,
    0.11580216127797223,0.22287798151967794,0.05991694265080245,
    0.1888225775300826,0.03891898167610717,0.05491866198626455,
    0.20340966480076852,0.2052254670613944
])

# Square the values
X2 = vals**2
X2_reshaped = X2.reshape(-1,1)

# Fit 1-component Gaussian mixture
mix1 = GaussianMixture(n_components=1, random_state=123)
mix1.fit(X2_reshaped)

# Fit 2-component Gaussian mixture
mix2 = GaussianMixture(n_components=2, random_state=123)
mix2.fit(X2_reshaped)

# Compute BIC and AIC
bic1 = mix1.bic(X2_reshaped)
bic2 = mix2.bic(X2_reshaped)
aic1 = mix1.aic(X2_reshaped)
aic2 = mix2.aic(X2_reshaped)

# Cluster assignments for 2-component mixture
clusters2 = mix2.predict(X2_reshaped)

# Proportion of points in cluster 2
prop2 = np.mean(clusters2 == 1)  # sklearn labels 0/1
cluster_balance = min(prop2, 1-prop2)

# Fill output array
out[0,0] = len(vals)
out[0,1] = bic1
out[0,2] = bic2
out[0,3] = aic1
out[0,4] = aic2
out[0,5] = cluster_balance

print(out)


[[  20.         -104.25565878 -117.76160009 -106.24712332 -122.74026146
     0.2       ]]


In [34]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, LongType

# Define output schema with cluster means, max_vals2, and maxMAF added
schema = StructType([
    StructField("variantId", StringType()),
    StructField("n_points", LongType()),
    StructField("bic1", DoubleType()),
    StructField("bic2", DoubleType()),
    StructField("aic1", DoubleType()),
    StructField("aic2", DoubleType()),
    StructField("cluster_balance", DoubleType()),
    StructField("cluster1_mean", DoubleType()),
    StructField("cluster2_mean", DoubleType()),
    StructField("max_vals2", DoubleType()),
    StructField("maxMAF", DoubleType())
])

def fit_gaussian_mixture(pdf: pd.DataFrame) -> pd.DataFrame:
    results = []
    # ensure variantId is a string
    variant_id = str(pdf["variantId"].iloc[0])
    vals = pdf["absEstimatedBeta"].dropna().values
    vals2 = vals ** 2
    
    # Calculate max of vals2
    max_vals2 = float(np.max(vals2)) if len(vals2) > 0 else np.nan
    
    # Calculate max of MAF
    maf_values = pdf["maf"].dropna().values
    max_maf = float(np.max(maf_values)) if len(maf_values) > 0 else np.nan

    if len(vals2) < 10:
        results.append([variant_id, len(vals2), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, max_vals2, max_maf])
    else:
        X2 = vals2.reshape(-1, 1)
        try:
            # Fit 1- and 2-component Gaussian mixtures
            mix1 = GaussianMixture(n_components=1, random_state=123).fit(X2)
            mix2 = GaussianMixture(n_components=2, random_state=123).fit(X2)

            # Compute AIC/BIC
            bic1, bic2 = mix1.bic(X2), mix2.bic(X2)
            aic1, aic2 = mix1.aic(X2), mix2.aic(X2)

            # Cluster balance
            clusters2 = mix2.predict(X2)
            prop2 = np.mean(clusters2 == 1)
            cluster_balance = min(prop2, 1 - prop2)

            # Extract cluster means from the 2-component mixture
            cluster1_mean = float(mix2.means_[0][0])
            cluster2_mean = float(mix2.means_[1][0])

            results.append([variant_id, len(vals2), bic1, bic2, aic1, aic2, cluster_balance, cluster1_mean, cluster2_mean, max_vals2, max_maf])

        except Exception:
            results.append([variant_id, len(vals2), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, max_vals2, max_maf])

    return pd.DataFrame(results, columns=[c.name for c in schema.fields])

# Apply grouped Pandas UDF
result_sdf = vlp_beta_max_high_ps.groupBy("variantId").applyInPandas(fit_gaussian_mixture, schema)

# Collect to pandas
result_df = result_sdf.toPandas()

print(result_df.head())

          variantId  n_points       bic1       bic2       aic1       aic2  \
0  10_112994312_T_C        16 -41.334811 -52.940810 -42.879988 -56.803754   
1  10_112995025_T_C        11 -37.636370 -37.303100 -38.432160 -39.292576   
2  10_112998590_C_T        52  92.424144 -46.297962  88.521657 -56.054181   
3    10_6052734_C_T        14 -33.607757 -61.393513 -34.885871 -64.588800   
4  11_116778201_G_C        21  15.086185  -4.056822  12.997140  -9.279434   

   cluster_balance  cluster1_mean  cluster2_mean  max_vals2    maxMAF  
0         0.500000       0.004935       0.080194   0.199758  0.293772  
1         0.363636       0.068489       0.015255   0.124536  0.293328  
2         0.019231       0.036446       3.896852   3.896852  0.293143  
3         0.142857       0.180886       0.019192   0.228003  0.089884  
4         0.047619       0.099144       1.286829   1.286829  0.137992  


In [35]:
result_df

,variantId,n_points,bic1,bic2,aic1,aic2,cluster_balance,cluster1_mean,cluster2_mean,max_vals2,maxMAF
0,10_112994312_T_C,16,-41.334811,-52.940810,-42.879988,-56.803754,0.500000,0.004935,0.080194,0.199758,0.293772
1,10_112995025_T_C,11,-37.636370,-37.303100,-38.432160,-39.292576,0.363636,0.068489,0.015255,0.124536,0.293328
2,10_112998590_C_T,52,92.424144,-46.297962,88.521657,-56.054181,0.019231,0.036446,3.896852,3.896852,0.293143
3,10_6052734_C_T,14,-33.607757,-61.393513,-34.885871,-64.588800,0.142857,0.180886,0.019192,0.228003,0.089884
4,11_116778201_G_C,21,15.086185,-4.056822,12.997140,-9.279434,0.047619,0.099144,1.286829,1.286829,0.137992
...,...,...,...,...,...,...,...,...,...,...,...
113,9_133274293_AC_A,12,-88.522244,-87.255805,-89.492057,-89.680338,0.333333,0.002913,0.011640,0.017171,0.202442
114,9_133274295_A_T,15,-31.729255,-60.200595,-33.145356,-63.740846,0.400000,0.005587,0.134466,0.199179,0.194176
115,9_22103814_A_G,12,-45.709796,-50.606293,-46.679609,-53.030826,0.083333,0.027593,0.114795,0.114795,0.490481
116,9_22124505_A_T,16,-95.397682,-103.909833,-96.942859,-107.772776,0.500000,0.003969,0.020996,0.036819,0.498646


In [36]:
sum(result_df["bic2"]<result_df["bic1"])

100

In [37]:
result_df["cluster_balance"].describe()

count    118.000000
mean       0.222058
std        0.151927
min        0.019231
25%        0.083333
50%        0.197856
75%        0.356061
max        0.500000
Name: cluster_balance, dtype: float64

In [38]:
100/118

0.847457627118644

In [39]:
result_df[result_df["cluster_balance"]==0.5]

,variantId,n_points,bic1,bic2,aic1,aic2,cluster_balance,cluster1_mean,cluster2_mean,max_vals2,maxMAF
0,10_112994312_T_C,16,-41.334811,-52.940810,-42.879988,-56.803754,0.5,0.004935,0.080194,0.199758,0.293772
5,11_1219991_G_T,12,38.786597,27.272622,37.816784,24.848089,0.5,0.047727,1.700959,2.971126,0.114920
45,19_10352442_G_C,18,-27.340383,-42.285845,-29.121127,-46.737704,0.5,0.162768,0.023078,0.366773,0.044501
116,9_22124505_A_T,16,-95.397682,-103.909833,-96.942859,-107.772776,0.5,0.003969,0.020996,0.036819,0.498646


In [26]:
vlp_beta_max_high_ps.filter(f.col("variantId")=="11_1219991_G_T").show(100)

+--------------+--------------------+--------------------+--------------------+------+--------------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------------+-------------------+-------------------+-------------+
|     variantId|             studyId|        studyLocusId|             variant|geneId|        originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|traitFromSourceMappedIds|   absEstimatedBeta|                maf|    diseaseId|
+--------------+--------------------+--------------------+--------------------+------+--------------------+---------------------+--------------------+-----------------+----------+---------------

In [27]:
result_df[result_df["bic2"]>result_df["bic1"]]


,variantId,n_points,bic1,bic2,aic1,aic2,cluster_balance,cluster1_mean,cluster2_mean,max_vals2
1,10_112995025_T_C,11,-37.636370,-37.303100,-38.432160,-39.292576,0.363636,0.068489,0.015255,0.124536
6,11_30204809_T_C,15,-74.562824,-71.704089,-75.978925,-75.244340,0.466667,0.038542,0.011952,0.065551
22,12_120978847_A_C,14,-133.035484,-126.577561,-134.313599,-129.772847,0.071429,0.005693,0.003313,0.008016
25,14_36269155_C_T,11,-5.077636,-3.955558,-5.873427,-5.945034,0.454545,0.336557,0.061098,0.473455
29,15_27985172_C_T,13,-56.051988,-52.908664,-57.181887,-55.733411,0.461538,0.041639,0.013519,0.095792
35,16_20381010_G_A,14,-60.820320,-58.500846,-62.098434,-61.696132,0.428571,0.011839,0.045499,0.087406
39,16_53769662_T_A,17,-122.840930,-122.700684,-124.507357,-126.866751,0.294118,0.003324,0.012030,0.022424
48,19_33235671_G_A,11,-74.400470,-74.328151,-75.196261,-76.317627,0.272727,0.008110,0.021603,0.024135
49,19_40847202_T_C,13,-96.653475,-91.729436,-97.783374,-94.554183,0.461538,0.014992,0.007315,0.020991
62,1_152313385_G_A,16,30.156168,33.237726,28.610990,29.374782,0.375000,0.894115,0.161287,1.974053


In [28]:
vlp_beta_max_high_ps.filter(f.col("variantId")=="6_160591981_T_C").show(100)

+---------------+--------------------+--------------------+--------------------+------+--------------------+---------------------+--------------------+-----------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------------+-------------------+--------------------+---------------+
|      variantId|             studyId|        studyLocusId|             variant|geneId|        originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|   majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|traitFromSourceMappedIds|   absEstimatedBeta|                 maf|      diseaseId|
+---------------+--------------------+--------------------+--------------------+------+--------------------+---------------------+--------------------+-----------------+----------+

In [29]:
import pandas as pd

# Calculate ratios
ratios = np.maximum(result_df["cluster1_mean"]/result_df["cluster2_mean"], 
                   result_df["cluster2_mean"]/result_df["cluster1_mean"])

print(f"Maximum ratio: {ratios.max()}")
print(f"Mean ratio: {ratios.mean()}")
print(f"Median ratio: {ratios.median()}")

Maximum ratio: 168.6534023283377
Mean ratio: 14.464289782864922
Median ratio: 7.675197417731421


In [30]:
import pandas as pd

# Calculate ratios
ratios = np.minimum(result_df["max_vals2"]/result_df["cluster2_mean"], 
                   result_df["max_vals2"]/result_df["cluster1_mean"])

print(f"Maximum ratio: {ratios.max()}")
print(f"Mean ratio: {ratios.mean()}")
print(f"Median ratio: {ratios.median()}")

Maximum ratio: 4.52507029053172
Mean ratio: 1.4458633767044238
Median ratio: 1.2253036240593245


In [31]:
ratios = np.maximum(result_df["max_vals2"]/result_df["cluster2_mean"], 
                   result_df["max_vals2"]/result_df["cluster1_mean"])
print(f"Maximum ratio: {ratios.max()}")
print(f"Mean ratio: {ratios.mean()}")
print(f"Median ratio: {ratios.median()}")

Maximum ratio: 168.65340232833807
Mean ratio: 19.21939524029146
Median ratio: 11.534427595590818


In [40]:
result_df

,variantId,n_points,bic1,bic2,aic1,aic2,cluster_balance,cluster1_mean,cluster2_mean,max_vals2,maxMAF
0,10_112994312_T_C,16,-41.334811,-52.940810,-42.879988,-56.803754,0.500000,0.004935,0.080194,0.199758,0.293772
1,10_112995025_T_C,11,-37.636370,-37.303100,-38.432160,-39.292576,0.363636,0.068489,0.015255,0.124536,0.293328
2,10_112998590_C_T,52,92.424144,-46.297962,88.521657,-56.054181,0.019231,0.036446,3.896852,3.896852,0.293143
3,10_6052734_C_T,14,-33.607757,-61.393513,-34.885871,-64.588800,0.142857,0.180886,0.019192,0.228003,0.089884
4,11_116778201_G_C,21,15.086185,-4.056822,12.997140,-9.279434,0.047619,0.099144,1.286829,1.286829,0.137992
...,...,...,...,...,...,...,...,...,...,...,...
113,9_133274293_AC_A,12,-88.522244,-87.255805,-89.492057,-89.680338,0.333333,0.002913,0.011640,0.017171,0.202442
114,9_133274295_A_T,15,-31.729255,-60.200595,-33.145356,-63.740846,0.400000,0.005587,0.134466,0.199179,0.194176
115,9_22103814_A_G,12,-45.709796,-50.606293,-46.679609,-53.030826,0.083333,0.027593,0.114795,0.114795,0.490481
116,9_22124505_A_T,16,-95.397682,-103.909833,-96.942859,-107.772776,0.500000,0.003969,0.020996,0.036819,0.498646


In [33]:
vlp_beta_max_high_ps.count()

2153

In [41]:
# Handle NaN values - only compute min if both values are not NaN
result_df['min_cluster_mean'] = result_df.apply(
    lambda row: min(row['cluster1_mean'], row['cluster2_mean']) 
    if pd.notna(row['cluster1_mean']) and pd.notna(row['cluster2_mean']) 
    else np.nan, 
    axis=1
)

In [42]:
result_df

,variantId,n_points,bic1,bic2,aic1,aic2,cluster_balance,cluster1_mean,cluster2_mean,max_vals2,maxMAF,min_cluster_mean
0,10_112994312_T_C,16,-41.334811,-52.940810,-42.879988,-56.803754,0.500000,0.004935,0.080194,0.199758,0.293772,0.004935
1,10_112995025_T_C,11,-37.636370,-37.303100,-38.432160,-39.292576,0.363636,0.068489,0.015255,0.124536,0.293328,0.015255
2,10_112998590_C_T,52,92.424144,-46.297962,88.521657,-56.054181,0.019231,0.036446,3.896852,3.896852,0.293143,0.036446
3,10_6052734_C_T,14,-33.607757,-61.393513,-34.885871,-64.588800,0.142857,0.180886,0.019192,0.228003,0.089884,0.019192
4,11_116778201_G_C,21,15.086185,-4.056822,12.997140,-9.279434,0.047619,0.099144,1.286829,1.286829,0.137992,0.099144
...,...,...,...,...,...,...,...,...,...,...,...,...
113,9_133274293_AC_A,12,-88.522244,-87.255805,-89.492057,-89.680338,0.333333,0.002913,0.011640,0.017171,0.202442,0.002913
114,9_133274295_A_T,15,-31.729255,-60.200595,-33.145356,-63.740846,0.400000,0.005587,0.134466,0.199179,0.194176,0.005587
115,9_22103814_A_G,12,-45.709796,-50.606293,-46.679609,-53.030826,0.083333,0.027593,0.114795,0.114795,0.490481,0.027593
116,9_22124505_A_T,16,-95.397682,-103.909833,-96.942859,-107.772776,0.500000,0.003969,0.020996,0.036819,0.498646,0.003969


In [43]:
result_df.to_csv("./data/gmm_fitting_results.csv",index=False)

In [68]:
result_df["cluster_balance"].describe()

count    118.000000
mean       0.222058
std        0.151927
min        0.019231
25%        0.083333
50%        0.197856
75%        0.356061
max        0.500000
Name: cluster_balance, dtype: float64

0      0.080194
1      0.068489
2      3.896852
3      0.180886
4      1.286829
         ...   
113    0.011640
114    0.134466
115    0.114795
116    0.020996
117    0.042205
Length: 118, dtype: float64

In [73]:
np.corrcoef(result_df["max_vals2"],np.maximum(result_df["cluster1_mean"],result_df["cluster2_mean"]))

array([[1.        , 0.86235068],
       [0.86235068, 1.        ]])

In [74]:
np.corrcoef(result_df["max_vals2"],np.minimum(result_df["cluster1_mean"],result_df["cluster2_mean"]))

array([[1.        , 0.64212054],
       [0.64212054, 1.        ]])

# Prediction for variants

In [44]:
vlp.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- variantId: string (nullable = true)
 |-- variant: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: integer (nullable = true)
 |    |-- end: integer (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- ref: string (nullable = true)
 |    |-- alt: string (nullable = true)
 |    |-- length: integer (nullable = true)
 |-- geneId: string (nullable = true)
 |-- originalBeta: double (nullable = true)
 |-- originalStandardError: double (nullable = true)
 |-- locusStatistics: struct (nullable = true)
 |    |-- locusSize: integer (nullable = true)
 |    |-- locusLength: integer (nullable = true)
 |    |-- locusStart: integer (nullable = true)
 |    |-- locusEnd: integer (nullable = true)
 |    |-- leadVariantPIP: double (nullable = true)
 |-- finemappingMethod: string (nullable = true)
 |-- isTransQtl: boolean (nullable = true)
 |-- variantEffect: a

In [45]:
vlp.select("diseaseId").distinct().count()

1403

In [46]:
# Alternative: guarantees exactly one row per diseaseId
window = Window.partitionBy("diseaseId").orderBy(f.desc("studyStatistics.nSamples"))

vlp_max_samples = vlp.withColumn("row_num", f.row_number().over(window)) \
                     .filter(f.col("row_num") == 1) \
                     .drop("row_num")

In [47]:
vlp_max_samples.count()

1403

In [48]:
vlp_max_samples.show(1)

+----------+--------------------+----------------+--------------------+------+------------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------------+------------------+-------------------+-----------+
|   studyId|        studyLocusId|       variantId|             variant|geneId|      originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|traitFromSourceMappedIds|  absEstimatedBeta|                maf|  diseaseId|
+----------+--------------------+----------------+--------------------+------+------------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+------------

In [49]:
diseaseIndex=vlp_max_samples.select(
    "diseaseId",
    "studyId",
    f.col("studyStatistics.nSamples").alias("nSamples"),
    f.col("rescaledStatistics.prevalence").alias("prevalence")
).cache()
diseaseIndex.count()

1403

In [50]:
diseaseIndex.show()

+-------------+--------------------+--------+--------------------+
|    diseaseId|             studyId|nSamples|          prevalence|
+-------------+--------------------+--------+--------------------+
|  EFO_0004254|          GCST010004|   11561| 0.32540437678401524|
|   HP_0000031|        GCST90478623|  409645|0.014807943463242563|
|MONDO_0009061|          GCST002025|    3059| 0.21052631578947367|
|  EFO_0006862|FINNGEN_R12_H8_ME...|  478519|0.007690394738766904|
|   HP_0001892|FINNGEN_R12_BLEEDING|  500348| 0.09798580188189021|
|MONDO_0005129|        GCST90271893|  818138| 0.04068629008798027|
|MONDO_0019640|        GCST90134254|   23859|0.005532503457814661|
|MONDO_0021116|        GCST90454345|  148877|   0.385553174768433|
|  EFO_0000571|        GCST90455519|  143595| 0.09670949545596992|
|  EFO_0000713|        GCST90018923|  626480|0.004622653556378495|
|  EFO_0004795|        GCST90077605|  422570| 0.17966017464562084|
|  EFO_0009485|        GCST90480612|  569533|0.003132390923791

In [51]:
diseaseIndex.toPandas().to_csv("./data/diseaseIndex.csv",index=False)

In [52]:
diseaseIndex=diseaseIndex.toPandas()

In [53]:
dind=diseaseIndex.copy()

In [54]:
def est_ps(beta,maxMAF,dind):
    from scipy.stats import ncx2
    #    (Intercept)            maxMAF         max_vals2  maxMAF:max_vals2  
    #     0.03798          -0.05967           0.05116          -0.14424  
    beta_small2=0.03798-0.05967*maxMAF+0.05116*(beta**2)-0.14424*maxMAF*(beta**2)
    #beta_small2=(beta**2)/11.534427595590818
    #beta_small2=(beta**2)/50
    dind["lin_beta2"]=beta_small2*((dind["prevalence"]*(1-dind["prevalence"]))**2)
    dind["r2"]=dind["lin_beta2"]*2*maxMAF*(1-maxMAF)
    dind["NCP"]=dind["r2"]*dind["nSamples"]
    p = ncx2.sf(32.84125, df=1, nc=dind["NCP"])
    obs_plei=sum(p)
    return obs_plei

In [57]:
beta=0.5
maxMAF=0.05
est_ps(beta,maxMAF,diseaseIndex)

125.98168901981343

In [59]:
varint_ps=pd.read_csv("/Users/yt4/Desktop/PSs/variants_with_fitness_scores.csv")

In [60]:
varint_ps["maxMAF"]

0        0.044689
1        0.393147
2        0.059694
3        0.356079
4        0.374352
           ...   
40701    0.232871
40702    0.116953
40703    0.490541
40704    0.013248
40705    0.422718
Name: maxMAF, Length: 40706, dtype: float64

In [61]:
# Apply est_ps function to each row
varint_ps['predicted_pleiotropy'] = varint_ps.apply(
    lambda row: est_ps(row['maxAbsBeta'], row['maxMAF'], diseaseIndex.copy()), 
    axis=1
)

In [62]:
varint_ps

,variantId,nonEURProportion,minAbsBeta,maxAbsBeta,minCoefficientDetermination,maxCoefficientDetermination,minEffectiveSampleSize,maxEffectiveSampleSize,minVarG,maxVarG,...,gerpNormalised,vepScore,fullModelDiseasesResid,biologicalFactorsDiseasesResid,statisticalFactorsDiseasesResid,sum_beta_variant,weighted_sum_beta_variant,count_rows,max_maf_variant,predicted_pleiotropy
0,10_100231378_T_A,1.000000,0.241801,0.241801,0.001248,0.001248,3.431020e+04,3.431020e+04,0.085383,0.085383,...,0.439000,0.10,0.073335,-0.360528,0.043313,0.241801,0.178117,1,0.044689,97.261355
1,10_100660841_T_G,1.000000,0.017299,0.017299,0.000036,0.000036,1.424218e+06,1.424218e+06,0.477165,0.477165,...,0.000000,0.00,-0.780858,-0.386125,-0.740000,0.017299,0.009946,1,0.393147,172.521840
2,10_100835542_T_A,0.000000,0.064451,0.064451,0.000117,0.000117,2.602102e+05,2.602102e+05,0.112262,0.112262,...,0.192667,0.00,-0.229954,-0.178591,-0.410599,0.064451,0.038091,1,0.059694,114.449621
3,10_102219168_T_A,1.000000,0.035843,0.035843,0.000147,0.000147,1.600105e+05,1.600105e+05,0.458574,0.458574,...,0.105000,0.00,-0.245786,-0.386458,-0.303356,0.035843,0.021488,1,0.356079,182.864322
4,10_102380845_G_A,1.000000,0.043559,0.043559,0.000222,0.000222,2.282206e+05,2.282206e+05,0.468425,0.468425,...,0.172917,0.10,-0.283624,-0.420798,-0.292639,0.043559,0.022696,1,0.374352,178.158499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40701,20_62357358_T_C,0.545455,0.035874,0.187506,0.000115,0.002549,1.374824e+04,3.012031e+05,0.289961,0.357285,...,0.000000,0.10,7.968783,8.587843,7.943598,1.244138,0.803380,10,0.232871,197.140371
40702,8_116618444_A_C,0.578947,0.055764,1.070688,0.000111,0.041044,1.848986e+03,4.739002e+05,0.143213,0.206550,...,0.910625,0.00,3.089929,9.280612,6.145730,0.599560,0.324768,12,0.116953,253.752933
40703,8_127401060_G_T,0.723404,0.029230,0.503670,0.000107,0.031699,1.848986e+03,5.218187e+05,0.187027,0.499821,...,0.900000,0.00,12.063827,16.698949,13.095563,2.479346,1.681755,19,0.490541,52.131894
40704,17_1880615_C_T,0.000000,0.207145,0.581642,0.000280,0.002211,1.886671e+04,3.679493e+05,0.026145,0.026145,...,0.810000,0.66,10.310941,10.302351,11.482443,4.829464,3.309419,13,0.013248,34.196148


In [63]:
# Fix the typo and use numpy.corrcoef or pandas.corr
np.corrcoef(varint_ps["predicted_pleiotropy"], varint_ps["uniqueDiseases"])[0, 1]

nan

In [64]:
varint_ps["uniqueDiseases"].describe()

count    40706.000000
mean         1.479610
std          1.544425
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         85.000000
Name: uniqueDiseases, dtype: float64

In [65]:
varint_ps["predicted_pleiotropy"].describe()

count    40668.000000
mean       160.678140
std         44.291446
min          0.000042
25%        144.182358
50%        175.462136
75%        191.386520
max        432.543384
Name: predicted_pleiotropy, dtype: float64

In [ ]:
varint_ps.to_csv("/Users/yt4/Desktop/PSs/variants_with_fitness_scores_pred.csv",index=False)

25/10/17 17:10:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 514060 ms exceeds timeout 120000 ms
25/10/17 17:10:06 WARN SparkContext: Killing executors is not supported by current scheduler.
25/10/17 17:10:09 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$